# Promptfoo prompt tests (local)

Runs a small suite of prompts through the model and checks each response against simple
assertions (contains a word, contains all of several words, ...). It is a quick sanity
check of model quality, and a template for your own prompt tests.

This notebook runs **locally**, straight against the model's endpoint — no EvalHub and no
OpenShift. It is self-contained: the model settings *and* the tests are in the
configuration cell below, and no job file is needed. It is the notebook equivalent of
`run-promptfoo-local.sh`.

**How it works:** [promptfoo](https://www.promptfoo.dev/) is run through `npx`. A config
is built from the settings below, promptfoo sends each prompt to the model, and each
response is graded against that test's assertions.

**Requires:** Node.js (`npx`) and network access to the model endpoint.

## 1. Configuration

Everything you might want to change lives in this one cell, including the tests.

In [ ]:
import json, os, subprocess, tempfile
from getpass import getpass
from pathlib import Path

# --- Model endpoint (from .env, or prompted for if there is no .env) ---
def load_env():
    """Read KEY=VALUE lines from a .env file into os.environ. Returns False if there is no file.

    Looks in the kernel's working directory, then ./notebooks (some Jupyter servers start the
    kernel in the repo's parent folder rather than next to the notebook)."""
    p = next((c for c in (Path(".env"), Path("notebooks/.env")) if c.is_file()), None)
    if p is None:
        return False
    for line in p.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            key, value = line.split("=", 1)
            os.environ[key.strip()] = value.strip().strip("\"'")
    return True

if load_env():
    print("Loaded settings from .env")
    missing = [k for k in ("OPENAI_BASE_URL", "OPENAI_MODEL") if not os.environ.get(k)]
    if missing:
        raise SystemExit(f"Please set {', '.join(missing)} in .env (see dot_env.template)")
else:
    print("No .env file found (copy dot_env.template to .env to skip these prompts)")
    os.environ["OPENAI_BASE_URL"] = input("Model base URL (e.g. https://host/v1): ").strip()
    os.environ["OPENAI_MODEL"] = input("Model name: ").strip()
    os.environ["OPENAI_API_KEY"] = getpass("API key (leave blank if none): ")

MODEL_URL = os.environ["OPENAI_BASE_URL"].rstrip("/")  # OpenAI-compatible base URL
MODEL_NAME = os.environ["OPENAI_MODEL"]
MODEL_API_KEY = os.environ.get("OPENAI_API_KEY", "")


# --- Proxy (optional; set PROXY_URL in .env, leave empty for a direct connection) ---
def apply_proxy():
    """Export the proxy settings in the form each tool underneath reads them.

    uv, Python (requests / huggingface_hub / aiohttp) and npm read HTTP(S)_PROXY and NO_PROXY
    (upper or lower case); Node's built-in fetch needs NODE_USE_ENV_PROXY; npm also has its own
    npm_config_* names. CA_BUNDLE is for proxies that re-sign HTTPS traffic with a company CA."""
    url, no_proxy = os.environ.get("PROXY_URL", "").strip(), os.environ.get("NO_PROXY", "").strip()
    if url:
        # The model endpoint is sent through the proxy too, unless its host is listed in NO_PROXY.
        for k in ("HTTP_PROXY", "HTTPS_PROXY", "http_proxy", "https_proxy", "npm_config_proxy", "npm_config_https_proxy"):
            os.environ[k] = url
        os.environ["NODE_USE_ENV_PROXY"] = "1"
        if no_proxy:
            for k in ("NO_PROXY", "no_proxy", "npm_config_noproxy"):
                os.environ[k] = no_proxy
        print("Using proxy", url.split("@")[-1], "(NO_PROXY: " + (no_proxy or "none") + ")")
    ca = os.environ.get("CA_BUNDLE", "").strip()
    if ca:
        ca = str(Path(ca).expanduser().resolve())
        for k in ("SSL_CERT_FILE", "REQUESTS_CA_BUNDLE", "CURL_CA_BUNDLE", "NODE_EXTRA_CA_CERTS"):
            os.environ[k] = ca
        print("Using CA bundle", ca)

apply_proxy()


# --- promptfoo ---
PROMPTFOO_VERSION = "0.123.1"  # same version the in-cluster adapter uses
CONCURRENCY = 4                # parallel requests to the model
THRESHOLD = 1.0                # minimum fraction of tests that must pass (1.0 = all)

# --- Prompts and tests ---
# Each prompt template is filled in with a test's `vars`; each response must satisfy
# every assertion on that test. Assertion types: https://www.promptfoo.dev/docs/configuration/expected-outputs/
PROMPTS = ["{{question}}"]
TESTS = [
    {"description": "Capital of France", "vars": {"question": "What is the capital of France?"},
     "assert": [{"type": "icontains", "value": "paris"}]},
    {"description": "Capital of Japan", "vars": {"question": "What is the capital of Japan?"},
     "assert": [{"type": "icontains", "value": "tokyo"}]},
    {"description": "Basic arithmetic", "vars": {"question": "What is 2 + 2?"},
     "assert": [{"type": "icontains", "value": "4"}]},
    {"description": "Mitochondria summary",
     "vars": {"question": "Summarize this in one sentence: The mitochondria is the powerhouse of the cell."},
     "assert": [{"type": "contains-all", "value": ["energy", "cell"]}]},
]


## Helper

`run()` launches a command and streams its output into the notebook as it arrives, so you
can watch progress. It raises an error if the command exits non-zero.

In [ ]:
def run(cmd, env=None):
    proc = subprocess.Popen(cmd, env={**os.environ, **(env or {})}, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="")
    if proc.wait() != 0:
        raise RuntimeError(f"{cmd[0]} exited with {proc.returncode}")

## 2. Run promptfoo

Writes a temporary promptfoo config (prompts, tests and the model as an OpenAI-compatible
provider) and runs `promptfoo eval`. Raw results go to `.eval-notebook-promptfoo.json`.
promptfoo exits non-zero when any test fails; that is expected, so it is reported and the
results are scored below.

In [ ]:
OUTPUT = Path(".eval-notebook-promptfoo.json")
config = {"prompts": PROMPTS, "tests": TESTS,
          "providers": [{"id": f"openai:chat:{MODEL_NAME}",
                         "config": {"apiBaseUrl": f"{MODEL_URL}", "apiKey": MODEL_API_KEY or "not-required"}}]}

with tempfile.TemporaryDirectory() as workdir:
    config_file = Path(workdir) / "config.json"
    config_file.write_text(json.dumps(config))
    try:
        run(["npx", "--yes", f"promptfoo@{PROMPTFOO_VERSION}", "eval", "-c", str(config_file),
             "-o", str(OUTPUT), "--no-cache", "--no-progress-bar", "-j", str(CONCURRENCY)])
    except RuntimeError as e:
        print(e)

## 3. Score and pass/fail

The pass rate is passed tests divided by all tests (failures and errors both count against it).

In [ ]:
stats = json.loads(OUTPUT.read_text())["results"]["stats"]
total = stats["successes"] + stats["failures"] + stats["errors"]
pass_rate = stats["successes"] / total if total else 0
print(f"passed {stats['successes']}, failed {stats['failures']}, errors {stats['errors']}")
print("PASS" if total and pass_rate >= THRESHOLD else "FAIL", f"- pass rate {pass_rate:.2f} vs threshold {THRESHOLD}")

## 4. Per-test breakdown

The question, the model's answer and — for failures — why the assertion failed.

In [ ]:
for r in json.loads(OUTPUT.read_text())["results"]["results"]:
    print(("PASS" if r["success"] else "FAIL"), "-", r["testCase"].get("description"))
    print("  Q:", r["vars"].get("question"))
    print("  A:", (r.get("response") or {}).get("output"))
    if not r["success"]:
        print("  Why:", (r.get("gradingResult") or {}).get("reason") or r.get("error"))